In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

In [2]:
from DFTStructureGenerator import B_N_Cl, mol_manipulation
import glob, os
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from tqdm import tqdm
import pandas as pd
import pickle

In [3]:
# DFT Method In Need
#p opt=(maxcycles=150) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm
OPT_METHOD = "opt freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm"
SPE_WFN_METHOD = "wb97xd/6-311+g(d,p) scrf=(smd,solvent=toluene) nosymm output=wfn"
OM_METHOD = "opt=(modredundant,maxcycles=100) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
TS_METHOD = "opt=(calcfc,ts,noeigen,maxcycles=100) freq b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"
SPE_SPECIAL = "b3lyp/6-31g(d) em=gd3bj scrf=(smd,solvent=toluene) nosymm"

In [4]:
root_file = "G:/work/B_Cl_Nu2/Data"
if not os.path.isdir(root_file):
    os.mkdir(root_file)
mol_xtb_file = os.path.join(root_file, 'Mol_xtb') 
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)
mol_dir = os.path.join(root_file, 'Mols')
dft_dir = os.path.join(root_file, 'GS_OPT')
spe_dir = os.path.join(root_file, 'GS_SPE')

# 1. Preprocess Reactants

In [ ]:
B_N_Cl.generate_combinations(
    reactant_file='Data/Reactants.csv',
    result_file='Data/Processed_Reactants.csv',
)

# 2. Optimization Structure

In [ ]:
B_N_Cl.B_N_Single_Xtb(root_file=root_file,
    result_file='Data/Processed_Reactants.csv', 
    mol_xtb_name = "Mol_xtb")

In [ ]:
all_strs = ['B', 'Other']
all_spins = [2,1]
for name, spin in zip(all_strs, all_spins):
    mol_xtb_file = os.path.join(root_file, 'Mol_xtb')
    root_dir = mol_xtb_file + "/" + name
    mol_dir = os.path.join(root_file, 'Mols')
    dft_dir = root_file + "/GS_OPT"
    if not os.path.isdir(dft_dir):
        os.mkdir(dft_dir)

    dft_dir_ = root_file + "/GS_OPT" + "/" + name
    if not os.path.isdir(dft_dir_):
        os.mkdir(dft_dir_)

    B_N_Cl.smiles_DFT_calc(root_dir, mol_dir, dft_dir_, method = OPT_METHOD, conf_limit = 10, SpinMultiplicity=spin) 

In [ ]:
mol_manipulation.error_improve(dft_dir, mol_dir, 'Other', 'dust_bin', improve_dir='Other_imp')

In [ ]:
B_N_Cl.SPE_DFT_calc(root_file, 'GS_OPT/B', 'GS_SPE/B', "Mols", method=SPE_METHOD)

In [ ]:
mol_manipulation.error_improve(spe_dir, mol_dir, 'Other', 'dust_bin', improve_dir='Other_imp')

In [ ]:
mol_manipulation.error_improve(spe_dir, mol_dir, 'B_N_p_new', 'dust_bin', improve_dir='B_N_p_new_imp')

In [ ]:
B_N_Cl.SPE_DFT_calc_wfn(root_file, 'GS_OPT', 'GS_SPE_wfn', method=SPE_WFN_METHOD)

In [ ]:
B_N_Cl.SPE_DFT_calc_wfn(root_file, 'GS_OPT/B_N_p_d', 'GS_SPE_wfn/B_N_p_d', method=SPE_WFN_METHOD)

In [ ]:
B_N_Cl.collection_dft_single(
    result_path = 'Data/Processed_Reactants.csv', 
    mol_dir=mol_dir, dft_dir=dft_dir, spe_dir=spe_dir)


# 3.Descriptors

## Fingerprint

In [9]:
react_df = pd.read_csv('Data/Processed_Reactants.csv')
all_smiles = list(set(react_df['Smiles'].to_list()))
B_df = react_df.loc[react_df['Type'] == "B"]
S_df = react_df.loc[react_df['Type'] == "S"]
ini_df = react_df.loc[react_df['Type'] == "ini"]
sol_df = react_df.loc[react_df['Type'] == "sol"]
eqs = [1.5, 2.0, 2.5, 3.0]
target_idx = np.array(np.meshgrid(np.arange(len(B_df['Index'])), np.arange(len(S_df['Index'])), np.arange(len(ini_df['Index'])), np.arange(len(sol_df['Index'])), np.arange(len(eqs)))).T.reshape(-1, 5)
B_mols = [Chem.MolFromSmiles(smi) for smi in B_df['Smiles']]
S_mols = [Chem.MolFromSmiles(smi) for smi in S_df['Smiles']]
ini_mols = [Chem.MolFromSmiles(smi) for smi in ini_df['Smiles']]
B_fps = np.array([np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)) for mol in B_mols])
S_fps = np.array([np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)) for mol in S_mols])
ini_fps = np.array([np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)) for mol in ini_mols])
sol_fps = np.eye(len(sol_df))
eqs_fps = np.eye(len(eqs))
B_dict = {B_df['Index'].iloc[i]: B_fps[i].tolist() for i in range(len(B_df))}
S_dict = {S_df['Index'].iloc[i]: S_fps[i].tolist() for i in range(len(S_df))}
ini_dict = {ini_df['Index'].iloc[i]: ini_fps[i].tolist() for i in range(len(ini_df))}
sol_dict = {sol_df['Index'].iloc[i]: sol_fps[i].tolist() for i in range(len(sol_df))}
final_dict = {**B_dict, **S_dict, **ini_dict, **sol_dict}


In [10]:
with open("Data/Fingerprint.pkl", 'wb') as f:
    pickle.dump(final_dict, f)

## PhysOrg

In [ ]:
root_file = "G:/work/B_Cl_Nu2/Data"
if not os.path.isdir(root_file):
    os.mkdir(root_file)
mol_xtb_file = os.path.join(root_file, 'Mol_xtb') 
if not os.path.isdir(mol_xtb_file):
    os.mkdir(mol_xtb_file)
mol_dir = os.path.join(root_file, 'Mols')
dft_dir = os.path.join(root_file, 'GS_OPT')
spe_dir = os.path.join(root_file, 'GS_SPE')

In [ ]:
from morfeus import BuriedVolume
SOL_KEY = {3016:9, 3017:6, 3018:3, 3019:1, 3020:0.33}
target_csv = pd.read_csv('Data/Processed_Reactants.csv')
index_map = {}
for line_id, line in tqdm(target_csv.iterrows()):
    pre_Index = int(line['Index'])
    if pre_Index >= 3016 and pre_Index <= 3020:
        Index = 3003
        line = target_csv.loc[target_csv['Index'] == Index].iloc[0]
    else:
        Index = pre_Index
    Smiles = line['Smiles']
    Atomid = int(line['Atomid'])
    conf = int(line['conf_idxs'])
    G_eng = line['G_energy']
    type_ = line['Type']
    
    mol = B_N_Cl.mol_manipulation.smiles2mol(Smiles)
    react_name = f"{Index:05}_{conf:04}.log"
    log = B_N_Cl.logfile_process.Logfile(dft_dir + "/" + react_name)
    mol = Chem.MolFromMolFile(mol_dir + "/" + f'{Index:05}.mol', removeHs=False, sanitize=False)
    
    # BNreact
    descriptor = [G_eng]
    descriptor += log.read_orbit_eng()
    descriptor += [log.get_dipole()]
    position = log.first_atom_position
    charges, spins = log.read_charge_spin_density()
    if type_ == "B":
        descriptor += [spins[Atomid]]
    elif type_ == "BNCl":
        another_Atomid = [each for each in mol.GetAtoms() if each.GetSymbol() == 'B'][0].GetIdx()
    elif type_ == "S":
        another_Atomid = [each for each in mol.GetAtomWithIdx(Atomid).GetNeighbors() if each.GetSymbol() != 'H'][0].GetIdx()
    elif type_ == 'ini':
        descriptor += [Atomid + 273.15]
        if mol.HasSubstructMatch(Chem.MolFromSmarts("N=N")):
            Atomid = mol.GetSubstructMatches(Chem.MolFromSmarts("N=N"))[0][0]
            another_Atomid = [each for each in mol.GetAtomWithIdx(Atomid).GetNeighbors() if each.GetSymbol() != 'N'][0].GetIdx()
        elif mol.HasSubstructMatch(Chem.MolFromSmarts("OO")):
            Atomid = mol.GetSubstructMatches(Chem.MolFromSmarts("OO"))[0][0]
            another_Atomid = [each for each in mol.GetAtomWithIdx(Atomid).GetNeighbors() if each.GetSymbol() != 'O'][0].GetIdx()
    elif type_ == 'sol':
        if pre_Index in SOL_KEY.keys():
            descriptor += [SOL_KEY[pre_Index]]
        else:
            descriptor += [0]
    if type_ not in ['B', 'sol']:
        descriptor += [charges[Atomid], charges[another_Atomid]]
        descriptor += [B_N_Cl.Tool.get_atoms_distance(position[Atomid], position[another_Atomid])]

        symbol_lists = log.symbol_list
        position = log.first_atom_position
        bv = BuriedVolume(symbol_lists, position, Atomid + 1, include_hs=1, radius=2, z_axis_atoms=[another_Atomid + 1], excluded_atoms=[another_Atomid + 1])
        bv.octant_analysis()
        descriptor += [np.sum(list(bv.octants['percent_buried_volume'].values())[:4])]
        bv = BuriedVolume(symbol_lists, position, Atomid + 1, include_hs=1, radius=4, z_axis_atoms=[another_Atomid + 1], excluded_atoms=[another_Atomid + 1])
        bv.octant_analysis()
        descriptor += [np.sum(list(bv.octants['percent_buried_volume'].values())[:4])]
        bv = BuriedVolume(symbol_lists, position, Atomid + 1, include_hs=1, radius=6, z_axis_atoms=[another_Atomid + 1], excluded_atoms=[another_Atomid + 1])
        bv.octant_analysis()
        descriptor += [np.sum(list(bv.octants['percent_buried_volume'].values())[:4])]
    index_map[pre_Index] = descriptor

In [ ]:
with open("Data/PhysOrgdes_new.pkl", 'wb') as f:
    pickle.dump(index_map, f)

In [ ]:
import pickle
with open("Data/Cldes_.pkl", 'rb') as f:
    Cl_des_map = pickle.load(f)
# Cl_00712, Cl_00713, Cl_00714
# Cl_des_map_new = {'Cl_05000':Cl_des_map['Cl_00712'], 'Cl_05001_Claid_00015':Cl_des_map['Cl_00713_Claid_00015'], 'Cl_05001_Claid_00019':Cl_des_map['Cl_00713_Claid_00019'], 'Cl_05002_Claid_00015':Cl_des_map['Cl_00714_Claid_00015'], 'Cl_05002_Claid_00017':Cl_des_map['Cl_00714_Claid_00017'],'Cl_05002_Claid_00020':Cl_des_map['Cl_00714_Claid_00020'] }
Cl_des_map_new = {'Cl_05000':Cl_des_map['Cl_00715'], 
                  'Cl_05001_Claid_00015':Cl_des_map['Cl_00716_Claid_00009'], 
                  'Cl_05001_Claid_00019':Cl_des_map['Cl_00716_Claid_00013'], 
                  'Cl_05002_Claid_00015':Cl_des_map['Cl_00717_Claid_00009'], 
                  'Cl_05002_Claid_00017':Cl_des_map['Cl_00717_Claid_00011'],
                  'Cl_05002_Claid_00020':Cl_des_map['Cl_00717_Claid_00014'],
                  'Cl_05003_Claid_00011':Cl_des_map['Cl_00720_Claid_00011'],
                  'Cl_05004_Claid_00009':Cl_des_map['Cl_00719_Claid_00009'],}
with open("Data/Cldes_new.pkl", 'wb') as f:
    pickle.dump(Cl_des_map_new, f)